<a href="https://colab.research.google.com/github/ankithprabhu96/Edukron2026/blob/main/Job_descriptions_resumes_similarities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers pypdf scikit-learn pandas

In [ ]:
import glob
import glob
import pickle
import numpy as np
import pandas as pd
import os

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:

def PDF_to_df(pdf_path:str)->pd.DataFrame:

  reader = PdfReader(pdf_path)

  job_texts = []

  for page in reader.pages:

      text = page.extract_text()

      if text:
          job_texts.append({
              'text':text.strip()
              })

  print("Job Descriptions:", len(job_texts))

In [ ]:
!pip install PyPDF2

In [ ]:
from pathlib import Path
import pandas as pd
from PyPDF2 import PdfReader


def extract_text_from_pdf(pdf_path: Path) -> str:
    """Extract text from one PDF file."""
    reader = PdfReader(str(pdf_path))

    pages_text = [
        page.extract_text() or ""
        for page in reader.pages
    ]

    return "\n".join(pages_text).strip()


def read_pdf_folder(folder_path: str) -> pd.DataFrame:
    """Read all PDF files from a folder and return filename and text."""
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    records = []

    for pdf_file in folder.glob("*.pdf"):
        try:
            records.append({
                "filename": pdf_file.name,
                "text": extract_text_from_pdf(pdf_file)
            })
        except Exception as e:
            records.append({
                "filename": pdf_file.name,
                "text": "",
                "error": str(e)
            })

    return pd.DataFrame(records)


def save_to_csv(df: pd.DataFrame, output_path: str) -> None:
    """Save DataFrame to CSV."""
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved successfully: {output_path}")


def main():
    folder_path = "/content/drive/MyDrive/Data Science/Resumes"
    output_path = "pdf_text_output.csv"

    df = read_pdf_folder(folder_path)

    print("Total PDFs processed:", len(df))
    print(df.head())

    save_to_csv(df, output_path)


if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
from PyPDF2 import PdfReader

pdf_path = "/content/drive/MyDrive/Data Science/20_Job_Descriptions.pdf"

reader = PdfReader(pdf_path)

data = []

for i,page in enumerate(reader.pages):

    page_text = page.extract_text() or ""

    lines = [line.strip() for line in page_text.split("\n") if line.strip()]


    # First line = Job Title
    name = lines[0]

    if i == 0:
      name = lines[1]

    # Remaining text = Description
    text = "\n".join(lines[1:])

    data.append({
        "name": name,
        "text": text
    })

df = pd.DataFrame(data)

print(df.shape)   # (20, 2)
print(df.head())

# Save if needed
df.to_csv("job_descriptions.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

# ==============================
# 1. Load CSV files
# ==============================

job_df = pd.read_csv("job_descriptions.csv")
resume_df = pd.read_csv("pdf_text_output.csv")

print(job_df.shape)
print(resume_df.shape)

# ==============================
# 2. Clean missing values
# ==============================

job_df["text"] = job_df["text"].fillna("")
resume_df["text"] = resume_df["text"].fillna("")

print("job_df", job_df.shape)

# ==============================
# 3. Load Hugging Face embedding model
# ==============================

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# ==============================
# 4. Generate embeddings
# ==============================

job_embeddings = model.encode(
    job_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

resume_embeddings = model.encode(
    resume_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Job Embeddings Shape:", job_embeddings.shape)
print("Resume Embeddings Shape:", resume_embeddings.shape)

# ==============================
# 5. Save embeddings as NPY files
# ==============================

np.save("job_embeddings.npy", job_embeddings)
np.save("resume_embeddings.npy", resume_embeddings)

print("Saved job_embeddings.npy")
print("Saved resume_embeddings.npy")

# ==============================
# 6. Optional: Save updated CSV also
# ==============================

job_df.to_csv("job_descriptions_clean.csv", index=False)
resume_df.to_csv("resumes_clean.csv", index=False)

print("Completed successfully")


import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# ==============================
# 1. Load CSV files
# ==============================

job_df = pd.read_csv("job_descriptions.csv")
resume_df = pd.read_csv("pdf_text_output.csv")

# ==============================
# 2. Load NPY embeddings
# ==============================

job_embeddings = np.load("job_embeddings.npy")
resume_embeddings = np.load("resume_embeddings.npy")

print("Job Embeddings:", job_embeddings.shape)
print("Resume Embeddings:", resume_embeddings.shape)

# ==============================
# 3. Calculate Similarity / Distance
# ==============================

cosine_scores = cosine_similarity(job_embeddings, resume_embeddings)

euclidean_scores = euclidean_distances(job_embeddings, resume_embeddings)

# ==============================
# 4. Get Top 10 Resume Matches
# ==============================

results = []

for job_index in range(len(job_df)):

    job_name = job_df.loc[job_index, "name"]

    # Cosine: higher score is better
    top_cosine_indexes = np.argsort(cosine_scores[job_index])[::-1][:10]

    for rank, resume_index in enumerate(top_cosine_indexes, start=1):
        results.append({
            "job_name": job_name,
            "method": "Cosine Similarity",
            "rank": rank,
            "resume_name": resume_df.loc[resume_index, "filename"],
            "score": cosine_scores[job_index][resume_index]
        })

    # Euclidean: lower distance is better
    top_euclidean_indexes = np.argsort(euclidean_scores[job_index])[:10]

    for rank, resume_index in enumerate(top_euclidean_indexes, start=1):
        results.append({
            "job_name": job_name,
            "method": "Euclidean Distance",
            "rank": rank,
            "resume_name": resume_df.loc[resume_index, "filename"],
            "score": euclidean_scores[job_index][resume_index]
        })

# ==============================
# 5. Convert to DataFrame
# ==============================

match_df = pd.DataFrame(results)

print(match_df.head(20))

# ==============================
# 6. Save Final Output
# ==============================

match_df.to_csv("job_resume_top10_matches.csv", index=False)

print("Saved: job_resume_top10_matches.csv")